## Weighted stats
- This notebook is made in the Programming for Data Analytics lecture on weighted descriptive statistics.
- We are looking at popultions in each county of ireland


In [1]:
import pandas as pd

In [2]:
url = "https://ws.cso.ie/public/api.restful/PxStat.Data.Cube_API.ReadDataset/FY006A/CSV/1.0/en"
df = pd.read_csv(url)
df.tail(3)

,STATISTIC,Statistic Label,TLIST(A1),CensusYear,C02199V02655,Sex,C02076V03371,Single Year of Age,C03789V04537,Administrative Counties,UNIT,VALUE
9789,FY006AC01,Population,2022,2022,2,Female,650,100 years and over,2ae19629-149d-13a3-e055-000000000001,Cavan County Council,Number,12
9790,FY006AC01,Population,2022,2022,2,Female,650,100 years and over,2ae19629-14a4-13a3-e055-000000000001,Donegal County Council,Number,31
9791,FY006AC01,Population,2022,2022,2,Female,650,100 years and over,2ae19629-1495-13a3-e055-000000000001,Monaghan County Council,Number,7


ok I notice that this has all the data for each gender I want to get rid of that

In [3]:
df = df[df["Sex"] != "Female"]
df = df[df["Sex"] != "Male"]
df.tail(3)

,STATISTIC,Statistic Label,TLIST(A1),CensusYear,C02199V02655,Sex,C02076V03371,Single Year of Age,C03789V04537,Administrative Counties,UNIT,VALUE
3261,FY006AC01,Population,2022,2022,-,Both sexes,650,100 years and over,2ae19629-149d-13a3-e055-000000000001,Cavan County Council,Number,18
3262,FY006AC01,Population,2022,2022,-,Both sexes,650,100 years and over,2ae19629-14a4-13a3-e055-000000000001,Donegal County Council,Number,33
3263,FY006AC01,Population,2022,2022,-,Both sexes,650,100 years and over,2ae19629-1495-13a3-e055-000000000001,Monaghan County Council,Number,8


Now I can do the code that is the prep for analysis.py  
I just noticed that there were more columns so I need to get their names

In [4]:
headers = df.columns.tolist()
headers

['STATISTIC',
 'Statistic Label',
 'TLIST(A1)',
 'CensusYear',
 'C02199V02655',
 'Sex',
 'C02076V03371',
 'Single Year of Age',
 'C03789V04537',
 'Administrative Counties',
 'UNIT',
 'VALUE']

In [5]:
drop_col_list = ['STATISTIC', 'Statistic Label','TLIST(A1)','CensusYear','C02199V02655','Sex','C02076V03371','C03789V04537','UNIT']
df.drop(columns=drop_col_list, inplace=True)
df = df[df["Single Year of Age"] != "All ages"]
df['Single Year of Age'] = df['Single Year of Age'].str.replace('Under 1 year', '0')
df['Single Year of Age'] = df['Single Year of Age'].str.replace('\D', '', regex=True)

df['Single Year of Age']=df['Single Year of Age'].astype('int64')
df['VALUE']=df['VALUE'].astype('int64')
#print (df.head(3))
df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 3232 entries, 32 to 3263
Data columns (total 3 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Single Year of Age       3232 non-null   int64 
 1   Administrative Counties  3232 non-null   object
 2   VALUE                    3232 non-null   int64 
dtypes: int64(2), object(1)
memory usage: 101.0+ KB


<>:5: SyntaxWarning: invalid escape sequence '\D'
<>:5: SyntaxWarning: invalid escape sequence '\D'
C:\Users\tihan\AppData\Local\Temp\ipykernel_14108\30480334.py:5: SyntaxWarning: invalid escape sequence '\D'
  df['Single Year of Age'] = df['Single Year of Age'].str.replace('\D', '', regex=True)


In [6]:
df_anal = pd.pivot_table(df, 'VALUE',"Single Year of Age","Administrative Counties")
print (df_anal.head(3))
# write out the entire file to local machine
df_anal.to_csv("population_for_analysis.csv")

Administrative Counties  Carlow County Council  Cavan County Council  \
Single Year of Age                                                     
0                                        699.0                1006.0   
1                                        649.0                1004.0   
2                                        689.0                1061.0   

Administrative Counties  Clare County Council  Cork City Council  \
Single Year of Age                                                 
0                                      1377.0             2283.0   
1                                      1360.0             2284.0   
2                                      1473.0             2306.0   

Administrative Counties  Cork County Council  Donegal County Council  \
Single Year of Age                                                     
0                                     4190.0                  1797.0   
1                                     4115.0                  1881.0   
2         

## Now we can do weighted descriptive statistics

In [7]:
headers = list(df_anal.columns)
district = headers[0]
district

'Carlow County Council'

Weighted mean is sum(age*population at age) / sum (populations at age)

In [8]:
number_people = df_anal[district].sum()
number_people


61968.0

In [9]:
df_anal

Administrative Counties,Carlow County Council,Cavan County Council,Clare County Council,Cork City Council,Cork County Council,Donegal County Council,Dublin City Council,Dún Laoghaire Rathdown County Council,Fingal County Council,Galway City Council,...,Monaghan County Council,Offaly County Council,Roscommon County Council,Sligo County Council,South Dublin County Council,Tipperary County Council,Waterford City & County Council,Westmeath County Council,Wexford County Council,Wicklow County Council
Single Year of Age,,,,,,,,,,,,,,,,,,,,,
0,699.0,1006.0,1377.0,2283.0,4190.0,1797.0,6213.0,2457.0,4009.0,792.0,...,868.0,935.0,748.0,743.0,3544.0,1932.0,1340.0,1129.0,1792.0,1710.0
1,649.0,1004.0,1360.0,2284.0,4115.0,1881.0,5699.0,2437.0,3894.0,757.0,...,821.0,937.0,764.0,766.0,3673.0,1808.0,1266.0,1158.0,1724.0,1682.0
2,689.0,1061.0,1473.0,2306.0,4290.0,1971.0,5830.0,2489.0,4308.0,792.0,...,870.0,945.0,893.0,764.0,3734.0,1836.0,1329.0,1108.0,1835.0,1796.0
3,754.0,1085.0,1468.0,2294.0,4437.0,2074.0,5580.0,2613.0,4384.0,799.0,...,900.0,1009.0,859.0,825.0,3856.0,2048.0,1476.0,1148.0,1864.0,1862.0
4,745.0,1047.0,1505.0,2243.0,4620.0,2089.0,5624.0,2658.0,4422.0,830.0,...,873.0,1007.0,833.0,820.0,3870.0,2072.0,1390.0,1208.0,1941.0,1897.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,11.0,23.0,40.0,54.0,84.0,48.0,200.0,95.0,48.0,19.0,...,17.0,19.0,28.0,21.0,41.0,41.0,26.0,24.0,42.0,38.0
97,11.0,13.0,31.0,39.0,65.0,27.0,115.0,57.0,51.0,12.0,...,9.0,16.0,13.0,22.0,38.0,33.0,20.0,23.0,37.0,25.0
98,7.0,8.0,15.0,23.0,47.0,15.0,81.0,45.0,21.0,15.0,...,11.0,8.0,7.0,12.0,17.0,21.0,17.0,11.0,22.0,16.0


In [10]:
cumages = df_anal[district].mul(df_anal.index, axis=0).sum()
cumages

2371825.0

In [11]:
weighted_mean = cumages/number_people
weighted_mean

38.27499677252776

#### Or you can use numpy

In [12]:
import numpy as np
w_mean = np.average(df_anal.index, weights=df_anal[district])
w_mean

38.27499677252776

### weighted median

    create a series of the cumlative sums and find the index of the middle value

In [13]:
cumsum = df_anal[district].cumsum()
cumsum

Single Year of Age
0        699.0
1       1348.0
2       2037.0
3       2791.0
4       3536.0
        ...   
96     61940.0
97     61951.0
98     61958.0
99     61965.0
100    61968.0
Name: Carlow County Council, Length: 101, dtype: float64

In [14]:
# sum of all the number of people
cutoff = df_anal[district].sum()/2
cutoff

30984.0

In [15]:
df_anal[district][cumsum>=cutoff].index[0]

39

## Weighted standard deviation

The is the same formula as the normal standard deviation, just with weights applied to the differenced to the weight mean

In [16]:
# incorrect way
df_anal[district].std()

306.958515918763

In [17]:
# incorrect
df_anal[district].mean()

613.5445544554456

We need the weighted std

In [18]:
import numpy as np
w_mean = np.average(df_anal.index, weights=df_anal[district])
w_mean

38.27499677252776

In [19]:
w_variance = np.average((df_anal.index - w_mean)**2, weights= df_anal[district])
w_variance

521.6941450425923

In [20]:
w_std = np.sqrt(w_variance)
w_std

22.84062488292718